# T4 — Haematopoietic stem and progenitor programs  [Reviewer 1 item 7]

Reviewer 1: *"I suggest the authors discuss a little about myeloid progenitors and myeloid stem
cell like HSC groups in gliomas. Just a few lines on how this could be that not necessary myeloid
programs but stem cell subsets with myeloid like states."*

Modules: Azimuth 2023 human bone-marrow reference (Hao et al. 2021), retrieved programmatically via
Enrichr. Scoring matched to the existing pipeline: `gseapy.ssgsea`, `sample_norm_method='rank'`,
on log2(TPM+1), n = 349.

## 1. ssGSEA and Kruskal-Wallis across ecotypes

In [ ]:
"""T4 (R1-7): haematopoietic stem / progenitor programs across immune ecotypes.
Modules: Azimuth 2023 human bone-marrow reference (Hao et al. 2021), retrieved via Enrichr.
Scoring matched to the existing pipeline: gseapy.ssgsea, sample_norm_method='rank', log2(TPM+1)."""
import numpy as np, pandas as pd, gseapy as gp, json
from scipy import stats

UP="/mnt/user-data/uploads/Open PBTA/Revision/Week1/_inputs"
WANT={ # Azimuth term -> module label
 "Bone Marrow-L2-Hematopoeitic Stem Cell":"HSC",
 "Bone Marrow-L2-Lymphoid Primed Multipotent Progenitor":"LMPP",
 "Bone Marrow-L2-Granulocyte Monocyte Progenitor":"GMP",
 "Bone Marrow-L2-Common Lymphoid Progenitor":"CLP",
 "Bone Marrow-L2-Erythroid Megakaryocyte Progenitor":"EMP (lineage control)",
 "Bone Marrow-L2-CD14 Monocyte":"CD14 monocyte (positive control)",
 "Bone Marrow-L2-Macrophage":"Macrophage (positive control)",
}
sets={}
for line in open("Azimuth_2023.gmt"):
    f=line.rstrip("\n").split("\t")
    if f[0] in WANT:
        sets[WANT[f[0]]]=[g.split(",")[0] for g in f[2:] if g.strip()]
for k,v in sets.items(): print(f"  {k:36} {len(v)} genes")

tpm=pd.read_csv(f"{UP}/tpm_for_cibersortx.tsv",sep="\t",index_col=0)
eco=pd.read_csv(f"{UP}/ecotype_LM22_main_k3_annotated.tsv",sep="\t").set_index("Kids_First_Biospecimen_ID")
tpm=tpm[[c for c in eco.index if c in tpm.columns]]
print("TPM restricted to main cohort:",tpm.shape)
for k,v in sets.items(): print(f"  {k:36} {sum(g in tpm.index for g in v)}/{len(v)} present")

log_tpm=np.log2(tpm.astype(float)+1.0)
res=gp.ssgsea(data=log_tpm,gene_sets=sets,sample_norm_method="rank",no_plot=True,
              threads=2,min_size=3,max_size=500,permutation_num=0,outdir=None)
s=res.res2d.copy(); s["NES"]=pd.to_numeric(s["NES"],errors="coerce")
s=s.pivot(index="Name",columns="Term",values="NES").astype(float)
s.index.name="Kids_First_Biospecimen_ID"
s=s.join(eco["ecotype"])
s.to_csv("T4_HSPC_ssGSEA_scores.tsv",sep="\t")
print("\nn scored:",len(s),"| ecotype counts:",s.ecotype.value_counts().to_dict())

ORDER=["Lymphocyte-inflamed","Myeloid-dominant","Immune-desert"]
mods=[m for m in sets]
rows=[]
for m in mods:
    grp=[s.loc[s.ecotype==e,m].values for e in ORDER]
    H,p=stats.kruskal(*grp)
    eps=(H-len(ORDER)+1)/(len(s)-len(ORDER))
    r=dict(module=m,KW_H=round(H,2),p=p,epsilon_sq=round(eps,3))
    for e,g in zip(ORDER,grp): r[f"mean_{e}"]=round(g.mean(),4)
    rows.append(r)
kw=pd.DataFrame(rows)
kw["q_BH"]=stats.false_discovery_control(kw.p)
kw=kw.sort_values("p")
print("\n=== Kruskal-Wallis across ecotypes ===")
print(kw[["module","KW_H","p","q_BH","epsilon_sq","mean_Lymphocyte-inflamed","mean_Myeloid-dominant","mean_Immune-desert"]].to_string(index=False,float_format=lambda x:f"{x:.4g}"))
kw.to_csv("T4_KW_by_ecotype.tsv",sep="\t",index=False)

# Dunn post hoc (BH within module)
def dunn(vals,labels):
    allv=np.concatenate(vals); rk=stats.rankdata(allv); ns=[len(v) for v in vals]
    idx=np.cumsum([0]+ns); N=len(allv)
    mr=[rk[idx[i]:idx[i+1]].mean() for i in range(len(vals))]
    _,cnt=np.unique(allv,return_counts=True); tie=(cnt**3-cnt).sum()
    sig2=(N*(N+1)/12)-tie/(12*(N-1))
    out=[]
    for i in range(len(vals)):
        for j in range(i+1,len(vals)):
            z=(mr[i]-mr[j])/np.sqrt(sig2*(1/ns[i]+1/ns[j]))
            out.append((labels[i],labels[j],z,2*stats.norm.sf(abs(z))))
    return out
drows=[]
for m in mods:
    vals=[s.loc[s.ecotype==e,m].values for e in ORDER]
    for a,b,z,p in dunn(vals,ORDER): drows.append(dict(module=m,group1=a,group2=b,z=round(z,2),p=p))
dn=pd.DataFrame(drows); dn["q_BH"]=stats.false_discovery_control(dn.p)
dn.to_csv("T4_Dunn_posthoc.tsv",sep="\t",index=False)
print("\n=== Dunn post hoc (BH across all 21 comparisons) ===")
print(dn.to_string(index=False,float_format=lambda x:f"{x:.3g}"))


Positive controls behave as expected (CD14 monocyte eps-sq = 0.476, macrophage 0.369,
both ordered Lymphocyte-inflamed > Myeloid-dominant > Immune-desert), which validates the scoring.

**GMP** shows a strong gradient (eps-sq = 0.205) but **HSC does not** (eps-sq = 0.018; Lymphocyte-inflamed
vs Myeloid-dominant z = 0.29, P = 0.775 — the two non-desert ecotypes are indistinguishable).

## 2. Specificity: is the progenitor signal independent of mature myeloid content?

In [ ]:
"""T4 specificity: is the GMP signal independent of mature monocyte/macrophage content?"""
import numpy as np, pandas as pd
from scipy import stats
sets={}
WANT={"Bone Marrow-L2-Hematopoeitic Stem Cell":"HSC","Bone Marrow-L2-Granulocyte Monocyte Progenitor":"GMP",
      "Bone Marrow-L2-CD14 Monocyte":"CD14 monocyte (positive control)","Bone Marrow-L2-Macrophage":"Macrophage (positive control)",
      "Bone Marrow-L2-Lymphoid Primed Multipotent Progenitor":"LMPP","Bone Marrow-L2-Common Lymphoid Progenitor":"CLP",
      "Bone Marrow-L2-Erythroid Megakaryocyte Progenitor":"EMP (lineage control)"}
for line in open("Azimuth_2023.gmt"):
    f=line.rstrip("\n").split("\t")
    if f[0] in WANT: sets[WANT[f[0]]]=[g.split(",")[0] for g in f[2:] if g.strip()]
print("=== module gene lists (Azimuth 2023 bone-marrow reference) ===")
for k in ["HSC","LMPP","GMP","CLP","EMP (lineage control)","CD14 monocyte (positive control)","Macrophage (positive control)"]:
    print(f"  {k:36} {', '.join(sets[k])}")
print("\ngene overlap GMP vs CD14 monocyte:",set(sets["GMP"])&set(sets["CD14 monocyte (positive control)"]) or "none")
print("gene overlap GMP vs Macrophage   :",set(sets["GMP"])&set(sets["Macrophage (positive control)"]) or "none")
print("gene overlap HSC vs GMP          :",set(sets["HSC"])&set(sets["GMP"]) or "none")

s=pd.read_csv("T4_HSPC_ssGSEA_scores.tsv",sep="\t",index_col=0)
ORDER=["Lymphocyte-inflamed","Myeloid-dominant","Immune-desert"]
mono="CD14 monocyte (positive control)"; mac="Macrophage (positive control)"
X=np.column_stack([np.ones(len(s)),s[mono].values,s[mac].values])
print("\n=== after residualising on mature monocyte + macrophage content ===")
rows=[]
for m in ["HSC","LMPP","GMP","CLP","EMP (lineage control)"]:
    y=s[m].values
    beta,*_=np.linalg.lstsq(X,y,rcond=None)
    r=y-X@beta
    grp=[r[s.ecotype.values==e] for e in ORDER]
    H,p=stats.kruskal(*grp); eps=(H-2)/(len(s)-3)
    raw=stats.kruskal(*[s.loc[s.ecotype==e,m].values for e in ORDER])
    rows.append(dict(module=m,eps_raw=round((raw[0]-2)/(len(s)-3),3),KW_H_resid=round(H,2),
                     p_resid=p,eps_resid=round(eps,3),
                     **{f"resid_mean_{e}":round(g.mean(),4) for e,g in zip(ORDER,grp)}))
out=pd.DataFrame(rows); out["q_BH_resid"]=stats.false_discovery_control(out.p_resid)
print(out.to_string(index=False,float_format=lambda x:f"{x:.4g}"))
out.to_csv("T4_residualised_specificity.tsv",sep="\t",index=False)


**No progenitor module survives adjustment for mature monocyte + macrophage content**
(all q >= 0.18). GMP's effect size collapses from 0.205 to 0.007 — a 29-fold drop — and the module gene
lists share **zero genes** with the monocyte and macrophage modules, so this is genuine covariance,
not gene overlap.

## 3. Figure

In [ ]:
import numpy as np, pandas as pd, matplotlib as mpl
mpl.use("Agg"); import matplotlib.pyplot as plt
mpl.rcParams.update({"font.family":"DejaVu Sans","font.size":8,"axes.linewidth":0.8,
                     "pdf.fonttype":42,"ps.fonttype":42})
s=pd.read_csv("T4_HSPC_ssGSEA_scores.tsv",sep="\t",index_col=0)
kw=pd.read_csv("T4_KW_by_ecotype.tsv",sep="\t").set_index("module")
rs=pd.read_csv("T4_residualised_specificity.tsv",sep="\t").set_index("module")
ORDER=["Lymphocyte-inflamed","Myeloid-dominant","Immune-desert"]
COL={"Lymphocyte-inflamed":"#3B82F6","Myeloid-dominant":"#EF4444","Immune-desert":"#7C8798"}
MODS=["HSC","LMPP","GMP","CLP","EMP (lineage control)","CD14 monocyte (positive control)"]
LAB=["HSC","LMPP","GMP","CLP","EMP\n(lineage control)","CD14 monocyte\n(positive control)"]

fig=plt.figure(figsize=(12.6,5.4),dpi=300)
gs=fig.add_gridspec(2,6,height_ratios=[1,.92],hspace=.62,wspace=.42)
for i,(m,lb) in enumerate(zip(MODS,LAB)):
    a=fig.add_subplot(gs[0,i])
    data=[s.loc[s.ecotype==e,m].values for e in ORDER]
    bp=a.boxplot(data,widths=.6,patch_artist=True,showfliers=False,
                 medianprops=dict(color="k",lw=1.0),whiskerprops=dict(lw=.7),capprops=dict(lw=.7))
    for p,e in zip(bp["boxes"],ORDER): p.set_facecolor(COL[e]); p.set_alpha(.55); p.set_edgecolor(COL[e]); p.set_lw(.8)
    a.set_xticks([1,2,3]); a.set_xticklabels(["LI","MD","ID"],fontsize=7)
    q=kw.loc[m,"q_BH"]; eps=kw.loc[m,"epsilon_sq"]
    a.set_title(f"{lb}\n$\\epsilon^2$ = {eps:.3f}   q = {q:.1e}",fontsize=7)
    if i==0: a.set_ylabel("ssGSEA NES")
    a.spines[["top","right"]].set_visible(False); a.tick_params(labelsize=7)

# bottom-left: effect size before vs after residualising
a=fig.add_subplot(gs[1,0:3])
order2=["HSC","LMPP","CLP","EMP (lineage control)","GMP"]
x=np.arange(len(order2)); w=.36
a.bar(x-w/2,[rs.loc[m,"eps_raw"] for m in order2],w,color="#B91C1C",label="unadjusted")
a.bar(x+w/2,[rs.loc[m,"eps_resid"] for m in order2],w,color="#94A3B8",
      label="residualised on mature\nmonocyte + macrophage content")
a.set_xticks(x); a.set_xticklabels(["HSC","LMPP","CLP","EMP","GMP"],fontsize=7.5)
a.set_ylabel("Kruskal-Wallis $\\epsilon^2$ across ecotypes")
a.set_title("F  Progenitor-like signal is not separable from mature myeloid content\n"
            "(no module survives adjustment; all q $\\geq$ 0.18)",fontsize=7.5,loc="left")
a.legend(fontsize=6.3,frameon=False,loc="upper left")
a.annotate("0.205 $\\rightarrow$ 0.007",xy=(4+w/2,.012),xytext=(2.55,.175),fontsize=6.8,color="#B91C1C",
           arrowprops=dict(arrowstyle="->",color="#B91C1C",lw=.8))
a.spines[["top","right"]].set_visible(False); a.tick_params(labelsize=7)

# bottom-right: GMP vs monocyte scatter
a=fig.add_subplot(gs[1,3:6])
for e in ORDER:
    sub=s[s.ecotype==e]
    a.scatter(sub["CD14 monocyte (positive control)"],sub["GMP"],s=9,c=COL[e],alpha=.65,lw=0,label=e)
r=np.corrcoef(s["CD14 monocyte (positive control)"],s["GMP"])[0,1]
a.set_xlabel("CD14 monocyte NES (mature myeloid content)"); a.set_ylabel("GMP NES")
a.set_title(f"G  GMP tracks mature monocyte content (Pearson r = {r:.2f})\n"
            "ecotypes separate along the shared axis, not across it",fontsize=7.5,loc="left")
a.legend(fontsize=6.3,frameon=False,markerscale=1.3,loc="upper left")
a.spines[["top","right"]].set_visible(False); a.tick_params(labelsize=7)

fig.suptitle("Haematopoietic stem and progenitor programs across immune ecotypes "
             "(Azimuth 2023 human bone-marrow reference; n = 349)",fontsize=9,y=.99)
for e in ("png","pdf"): fig.savefig(f"FigureS_HSPC_programs.{e}",dpi=300,bbox_inches="tight")
print("saved; GMP~monocyte r =",round(r,3))


## Interpretation, and what to write

The honest answer to the reviewer is a qualified no, which is stronger than agreeing:

> Committed granulocyte-monocyte progenitor programs do vary across ecotypes, but in bulk data that
> variation is statistically inseparable from mature monocyte and macrophage content, and uncommitted
> HSC programs do not distinguish the two non-desert ecotypes at all. We therefore cannot support --
> or exclude -- a distinct myeloid-progenitor or HSC-like subset as a driver of the ecotype gradient.
> Resolving this requires single-cell resolution.

Two limitations to state explicitly: the Azimuth L2 marker modules contain only 9 genes each, and
bulk deconvolution cannot separate a tumour-intrinsic stem-like program from haematopoietic
progenitor infiltration. This result is also the principled motivation for the single-cell work
requested in Reviewer 1 item 5.